In [4]:
import sys
import json
# TO CHANGE
BASEDIR = "../../.."
sys.path.insert(0, BASEDIR)

In [5]:
from pprint import pprint
import joblib

In [6]:
from src import PersonalAI, PersonalAIConfig
from src.kg_model import KnowledgeGraphModelConfig
from src.db_drivers.vector_driver.embedders import EmbedderModelConfig
from src.agents import AgentDriverConfig
from src.agents.utils import AgentConnectorConfig

from src.pipelines.qa import QAPipelineConfig
from src.pipelines.qa.kg_reasoning import KnowledgeGraphReasonerConfig
from src.pipelines.qa.query_preprocessing import QueryPreprocessorConfig
from src.pipelines.qa.query_preprocessing.decomposition import QueryDecomposerConfig
from src.pipelines.qa.query_preprocessing.denoising import QueryDenoiserConfig
from src.pipelines.qa.query_preprocessing.enhancing import QueryEnhancerConfig
from src.pipelines.qa.kg_reasoning.medium_reasoner import MediumKGReasonerConfig
from src.pipelines.qa.answers_aggregation import AnswersAggregatorConfig

from src.pipelines.qa.knowledge_retriever import KnowledgeRetrieverConfig
from src.pipelines.qa.knowledge_retriever.traversal_methods import WaterCirclesSearchConfig

from src.db_drivers.kv_driver.configs import DEFAULT_MIXEDKV_CONFIG
from src.db_drivers.kv_driver import KeyValueDriverConfig
from src.utils.logger import LogLevel, Logger
from src.utils import ModuleType

/home/dzigen/miniconda3/envs/pai/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
kg_config = KnowledgeGraphModelConfig(
    nodestree_config=None # Модель дерева вершин строиться не будет
)
EMBEDDER_MODEL_PATH = '../../../models/intfloat/multilingual-e5-large' # PATH TO APPROPRIATE EMBEDDER-MODEL
kg_config.embedders_configs['default_embedder'] = EmbedderModelConfig(
    model_name_or_path=EMBEDDER_MODEL_PATH
)
kg_config.agents_configs['default_llmagent'] = AgentDriverConfig(
    name = 'ollama',
    agent_config = AgentConnectorConfig(
        credentials= {'model': 'qwen2.5:14b', 'host': 'localhost', 'port': 11437}
    )
)

In [8]:
pai_config = PersonalAIConfig(
    lang='ru',
    log_level=LogLevel.DEBUG,
    kg_model_config=kg_config,
    qa_pipeline_config=QAPipelineConfig(
        preprocessor_config=QueryPreprocessorConfig(
            denoising_config=QueryDenoiserConfig(),
            enhancing_config=None,
            decomposition_config=QueryDecomposerConfig()
        ),
        reasoner_config=KnowledgeGraphReasonerConfig(
            reasoner_name='medium',
            reasoner_config=MediumKGReasonerConfig(
                enable_searchplan_steps_check=False,
                planlimit_exceeded_behaviour='casual_answer'
            )
        ),
        aggregator_config=AnswersAggregatorConfig(
            answers_summ_behaviour='casual_answer'
        )
    )
)

In [9]:
personalai = PersonalAI(pai_config)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 12655.67it/s]


In [10]:
response, stat = personalai.kg_model.AVAILABLE_AGENTS['default_llmagent'].generate("What is the purpose of life?")
print(response)

The purpose of life is a question that has puzzled philosophers, theologians, and individuals for centuries. Different people have different beliefs about what gives their lives meaning and purpose.

Some believe that the purpose of life is to seek happiness and fulfillment through personal growth, relationships with others, and contributing positively to society. Others may see the purpose of life as being in service to a higher power or following religious teachings and commandments.

There are also those who believe that there is no inherent meaning or purpose to life, and that individuals must create their own sense of purpose through their actions and choices.

Ultimately, the answer to this question is deeply personal and subjective. It's up to each individual to reflect on what they value most in life and determine for themselves what gives their existence meaning and direction.


In [11]:
def load_file(filepath: str) -> str:
    with open(filepath, 'r', encoding='utf-8') as fd:
        data = fd.read()
    return data

TEXT_EXAMPLES = [
    f"[FILENAME]: aggregate.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/aggregate.md")}",
    f"[FILENAME]: compare_people.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/compare_people.md")}",
    f"[FILENAME]: errors.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/errors.md")}",
    f"[FILENAME]: filters.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/filters.md")}",
    f"[FILENAME]: getting_started.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/getting_started.md")}",
    f"[FILENAME]: history.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/history.md")}",
    f"[FILENAME]: limits.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/limits.md")}",
    f"[FILENAME]: param_metrics.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/param_metrics.md")}",
    f"[FILENAME]: rows.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/rows.md")}",
    f"[FILENAME]: rules.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/rules.md")}",
    f"[FILENAME]: top_n.md\n[CONTENTS]:\n\n\n{load_file("../../../../smoking-room/b2e-synt-data/heimdall-skills/general/top_n.md")}",
]
print(len(TEXT_EXAMPLES))

11


In [13]:
extracted_triplets = []
for i, example in enumerate(TEXT_EXAMPLES):
    print(f"{i+1}. {example}")
    update_results, _, _ = personalai.update_memory(example)
    tmp_text_id, tmp_extracted_triplets = update_results
    extracted_triplets += tmp_extracted_triplets
    print("text_id: ", tmp_text_id)
    print("extracted triplets: ", len(tmp_extracted_triplets))

1. [FILENAME]: aggregate.md
[CONTENTS]:


---
name: aggregate
title: Агрегаты с группировкой (режим aggregate)
kind: reference
version: 1.0.0
status: active
domain: general
order: 50
description: Как посчитать метрики каталога с группировкой по колонкам. Когда нужен счёт или среднее, а не перечень строк.
tags: [агрегат, группировка, метрика, group by, count]
related:
  - {name: rows, relation: alternative, note: если нужен перечень}
  - {name: param_metrics, relation: follow_up, note: метрики с аргументами}
  - {name: getting_started, relation: prerequisite, note: поток запроса}
---

# Агрегаты с группировкой

Режим включается **наличием `metrics` или `param_metrics`** при отсутствии
`time_dimensions`. Отдельного поля режима нет — это частый источник ошибок:
запрос выглядит агрегатом, а возвращает строки, потому что `metrics` забыли.

```json
{
  "schema": "dm_core",
  "logic_model": "employee_actual",
  "columns": [
    "company"
  ],
  "metrics": [
    "fact_count"
  ],
  "order_by":

In [14]:
print(json.dumps(personalai.kg_model.count_items(detailed=True),indent=5))
personalai.kg_model.check_consistency()

{
     "graph_info": {
          "triplets": {
               "simple": 212,
               "hyper": 591,
               "episodic": 844,
               "time": 0
          },
          "nodes": {
               "object": 537,
               "hyper": 162,
               "episodic": 11,
               "time": 0
          }
     },
     "embeddings_info": {
          "nodes": {
               "object": {
                    "dense_nodes": 537,
                    "bm25_nodes": 537
               },
               "hyper": {
                    "dense_nodes": 162,
                    "bm25_nodes": 162
               },
               "episodic": {
                    "dense_nodes": 11,
                    "bm25_nodes": 11
               },
               "time": {
                    "dense_nodes": 0,
                    "bm25_nodes": 0
               }
          },
          "triplets": {
               "dense_triplets": 383,
               "bm25_triplets": 383
          }
     },
     "

True

In [15]:
personalai.textid_store.count_items()

{'textid_to_tripletsid': 11, 'tripletid_to_textsid': 1647}

In [16]:
print("mem kv_cache info:")
print(json.dumps(personalai.mem_pipeline.get_cache_stat(), indent=5))
print("mem agent tgen info:")
print(json.dumps(personalai.mem_pipeline.get_agent_tgen_stat(), indent=5))

mem kv_cache info:
{
     "MemPipeline": null,
     "extractor": {
          "LLMExtractor": null,
          "triplets_extraction_solver": 9,
          "thesises_extraction_solver": 11
     },
     "updator": {
          "LLMUpdator": null,
          "replace_simple_solver": 0,
          "replace_hyper_solver": 0
     }
}
mem agent tgen info:
{
     "extractor": {
          "triplets_extraction_solver": {
               "prompt_tokens_amount": {
                    "count": 9,
                    "count_not_null": 9,
                    "min": 1484,
                    "max": 2514,
                    "median": 1654,
                    "mean": 1794.556,
                    "std": 316.256,
                    "sum": 16151
               },
               "generated_tokens_amount": {
                    "count": 9,
                    "count_not_null": 9,
                    "min": 165,
                    "max": 880,
                    "median": 317,
                    "mean": 361.22

In [17]:
answer1, rinfo1, trace1 = personalai.answer_question("Сколько витрин (view) находятся в системе и в скольких схемах они распределены?")
pprint(answer1)

('В системе находится 37 витрин. Распределение по схемам:\n'
 '- sset - 1\n'
 '- technical - 1\n'
 '- stable - 3\n'
 '- dm_core - 14\n'
 '- dm_special - 7\n'
 '- recruitment - 4\n'
 '- anagent - 7')


In [18]:
answer1, rinfo1, trace1 = personalai.answer_question("В какой схеме расположено наибольшее количество витрин и сколько их там?")
pprint(answer1)

'Наибольшее количество витрин расположено в схеме dm_core - там их 14.'


In [19]:
answer1, rinfo1, trace1 = personalai.answer_question("Что вернёт сервер, если в теле запроса не будет ни time_dimensions, ни metrics, ни columns?")
pprint(answer1)

('Сервер вернёт ошибку query-empty при отсутствии параметров time_dimensions, '
 'metrics и columns в теле запроса.')


In [22]:
answer1, rinfo1, trace1 = personalai.answer_question("Какой метод позволяет быстро найти готовый рецепт под задачу?")
pprint(answer1)

('Чтобы быстро найти готовый рецепт, используйте функцию '
 'find_skills("<задача>"), например: find_skills("поиск рецептов с '
 'определенными ингредиентами"). Также можно применять фильтры и параметр '
 'has_embedding = true для уточнения поиска.')


In [23]:
answer1, rinfo1, trace1 = personalai.answer_question("Как получить обзор всех доступных доменов и приёмов?")
pprint(answer1)

('Для получения обзора всех доступных доменов и методов используйте функцию '
 'get_overview(). Эта функция предоставляет информацию о доступных доменах, '
 'методах, метриках и колонках через describe_model. Для её успешного '
 'использования требуется наличие соответствующих action_ids токена и '
 'allowed_groups модели; при отсутствии — будет возникать ошибка forbidden.')


In [24]:
personalai.close_connections()
del personalai